In [0]:
%python
#I needed it to clean the wrong schema
dbutils.fs.rm('/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/bronze_rides', True)
dbutils.fs.rm('/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/bronze_drivers', True)
dbutils.fs.rm('/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/silver_rides', True)
dbutils.fs.rm('/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/silver_drivers', True)
dbutils.fs.rm('/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/gold_preaggregated', True)
dbutils.fs.rm('/Volumes/workspace/streaming_cdc_test/raw_data/rides_schema_bronze', True)
dbutils.fs.rm('/Volumes/workspace/streaming_cdc_test/raw_data/drivers_schema_bronze', True)
dbutils.fs.rm('/Volumes/workspace/streaming_cdc_test/raw_data/drivers', True)
dbutils.fs.rm('/Volumes/workspace/streaming_cdc_test/raw_data/rides', True)

In [0]:
--I needed it to clean the wrong schema
DROP TABLE IF EXISTS workspace.streaming_cdc_test.drivers_bronze;
DROP TABLE IF EXISTS workspace.streaming_cdc_test.rides_bronze;
DROP TABLE IF EXISTS workspace.streaming_cdc_test.drivers_silver;
DROP TABLE IF EXISTS workspace.streaming_cdc_test.rides_silver;
DROP TABLE IF EXISTS workspace.streaming_cdc_test.preaggregated_gold

Bronze layer ingestion for drivers

In [0]:
%python
path = '/Volumes/workspace/streaming_cdc_test/raw_data/drivers'

streamingInputDriversDF = (
    spark.readStream
     .format("cloudFiles")
     .option("cloudFiles.format", "json")
     .option("cloudFiles.inferColumnTypes", "true")
     .option("cloudFiles.schemaLocation", '/Volumes/workspace/streaming_cdc_test/raw_data/drivers_schema_bronze')
     .load(path)
)

streamingInputDriversDF.writeStream\
                .option("checkpointLocation", '/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/bronze_drivers')\
                .option('mergeSchema', 'true')\
                .outputMode('append')\
                .trigger(availableNow=True)\
                .toTable('workspace.streaming_cdc_test.drivers_bronze')

Bronze layer ingestion for rides

In [0]:
%python
path = '/Volumes/workspace/streaming_cdc_test/raw_data/rides'

streamingInputRidesDF = (
    spark.readStream
     .format("cloudFiles")
     .option("cloudFiles.format", "json")
     .option("cloudFiles.inferColumnTypes", "true")
     .option("cloudFiles.schemaEvolutionMode", 'addNewColumns')
     .option("cloudFiles.schemaLocation", '/Volumes/workspace/streaming_cdc_test/raw_data/rides_schema_bronze')
     .load(path)
)

streamingInputRidesDF.writeStream\
                .option("checkpointLocation", '/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/bronze_rides')\
                .option('mergeSchema', 'true')\
                .outputMode('append')\
                .trigger(availableNow=True)\
                .toTable('workspace.streaming_cdc_test.rides_bronze')


Silver layer transformations for drivers. CDF is enabled for the drivers table to simulate real world-like data, as for example rank or experience of a driver can change. Deduplication on the driver_id is performed to avoid duplicates.

In [0]:
%python
import pyspark.sql.functions as F

def merge_drivers(df, batch_id):
    df_dedup = df.dropDuplicates(['id'])
    df_dedup.createOrReplaceTempView('drivers_micro_df')
    #MERGE WITH SCHEMA EVOLUTION is analogous to spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true") but for serverless
    sql_query = """
        MERGE WITH SCHEMA EVOLUTION INTO workspace.streaming_cdc_test.drivers_silver t
        USING drivers_micro_df s
        ON t.id = s.id
        WHEN MATCHED THEN
        UPDATE SET *
        WHEN NOT MATCHED
        THEN INSERT *
    """
    df_dedup.sparkSession.sql(sql_query)


spark.sql("""
          CREATE TABLE IF NOT EXISTS workspace.streaming_cdc_test.drivers_silver(
              id STRING,
              first_name STRING,
              last_name STRING,
              car_number STRING,
              experience INTEGER,
              rating INTEGER
          )
          TBLPROPERTIES (delta.enableChangeDataFeed = true)""")

streamingInputDriversDF = (spark.readStream
                        .table('workspace.streaming_cdc_test.drivers_bronze')
                        .withColumn('first_name', F.split(F.col('name'), ' ')[0])
                        .withColumn('last_name',  F.split(F.col('name'), ' ')[1])
                        .select('id', 'first_name', 'last_name', 'car_number', 'experience', 'rating')
)

(
    streamingInputDriversDF.writeStream
                    .format('delta')
                    .option('checkpointLocation', '/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/silver_drivers')
                    .outputMode("append")
                    .foreachBatch(merge_drivers)
                    .trigger(availableNow=True)
                    .start()
)
     

Silver layer transformation and ingestion for rides. Only appends data to rides. Ignores changes from the bronze layer, as rides are treated as append only.

In [0]:
%python
def merge_rides(df, batch_id):
    df_dedup = df.dropDuplicates(['driver_id'])
    df_drivers = df.sparkSession.read.table('workspace.streaming_cdc_test.drivers_silver')
    df_dedup = df_dedup.join(df_drivers, df_drivers.id == df_dedup.driver_id, "semi").select('ride_id', 'driver_id', 'distance', 'cost')
    df_dedup.createOrReplaceTempView('rides_micro_df')

    sql_query = """
        MERGE WITH SCHEMA EVOLUTION INTO workspace.streaming_cdc_test.rides_silver t
        USING rides_micro_df s
        ON t.ride_id = s.ride_id
        WHEN NOT MATCHED
        THEN INSERT *
    """
    df_dedup.sparkSession.sql(sql_query)

spark.sql("""
          CREATE TABLE IF NOT EXISTS workspace.streaming_cdc_test.rides_silver(
              ride_id STRING,
              driver_id STRING,
              distance INTEGER,
              cost INTEGER
          )
          """)

streamingInputRidesDF = (
    spark.readStream
         .table('workspace.streaming_cdc_test.rides_bronze')
         .select('ride_id', 'driver_id', 'distance', 'cost')
)

(
    streamingInputRidesDF.writeStream
                    .format('delta')
                    .option('checkpointLocation', '/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/silver_rides')
                    .outputMode("append")
                    .foreachBatch(merge_rides)
                    .trigger(availableNow=True)
                    .start()
)

Gold layer ingestion and transformation. Gold layer in this case is a preaggregate. Stateful streaming is used. drivers_silver table is read as a regular table to enrich the data from rides table with first_name and last_name of the driver. Liquid Clustering on driver_id is used, because it is a high cardinality column that is(likely) will be frequently used.

In [0]:
%python
import pyspark.sql.functions as F
def merge_gold_drivers(df, batch_id):
    df_drivers = df.sparkSession.read.table('workspace.streaming_cdc_test.drivers_silver')
    df_micro_preaggregate = df.join(df_drivers, df_drivers.id == df.driver_id, "inner")\
                              .select('driver_id', 'first_name', 'last_name', 'avg_distance', 'avg_cost', 'total_distance')        
    
    df_micro_preaggregate.createOrReplaceTempView('gold_preaggregate_micro_df')
    sql_query = """
        MERGE WITH SCHEMA EVOLUTION INTO workspace.streaming_cdc_test.preaggregated_gold t
        USING gold_preaggregate_micro_df s
        ON t.driver_id = s.driver_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """
    df_micro_preaggregate.sparkSession.sql(sql_query)

spark.sql("""
          CREATE TABLE IF NOT EXISTS workspace.streaming_cdc_test.preaggregated_gold(
              driver_id STRING,
              first_name STRING,
              last_name STRING,
              avg_distance DOUBLE,
              avg_cost DOUBLE,
              total_distance INTEGER
          )
          CLUSTER BY(driver_id)""")

streamingInputRidesDF = (
    spark.readStream
         .format('delta')
         .table('workspace.streaming_cdc_test.rides_silver')
)

#will maintain the state across batches
aggregatedRidesDF = (
    streamingInputRidesDF.groupBy('driver_id')
                         .agg(F.avg('distance').alias('avg_distance'),
                              F.avg('cost').alias('avg_cost'),
                              F.sum('distance').alias('total_distance'))
)

(
    aggregatedRidesDF.writeStream
                    .format('delta')
                    .option('checkpointLocation', '/Volumes/workspace/streaming_cdc_test/raw_data/checkpoints/gold_preaggregated')
                    .outputMode("update")
                    .trigger(availableNow=True)
                    .foreachBatch(merge_gold_drivers)
                    .start()
)